Here will be a jupyter notebook for the FV analysis.

In [1]:
from cobra.flux_analysis import flux_variability_analysis
import jupyter_utils as ju

In [13]:
base_dir = Path.home() / "Documents" / "PhD" / "10-19 Research" / "11 Data" / "11.09_Models"
model = ju.load_model(base_dir,"Experiment", "250620_iABA974.sbml")

#### Amino acid sequence analyzer

The **A, C, D, etc.** are the **one-letter codes** for the 20 standard amino acids in proteins.

Here’s the full table for reference for BLG:

| Code  | Amino Acid Name | 
| ----- | --------------- | 
| **A** | Alanine = 8.12%        | 
| **C** | Cysteine = 1.28%        |
| **D** | Aspartic acid = 4.70 %  |
| **E** | Glutamic acid = 5.56 %  |
| **F** | Phenylalanine = 3.42 %   |
| **G** | Glycine = 9.40 %         |
| **H** | Histidine = 2.99 %       |
| **I** | Isoleucine  = 2.14 %     |
| **K** | Lysine = 2.56 %          |
| **L** | Leucine = 15.81 %         |
| **M** | Methionine = 1.28 %      |
| **N** | Asparagine = 0.28 %     |
| **P** | Proline = 6.84 %         |
| **Q** | Glutamine = 6.41 %       |
| **R** | Arginine = 3.85 %        |
| **S** | Serine = 7.26 %          |
| **T** | Threonine = 4.70 %      |
| **V** | Valine = 7.69 %         |
| **W** | Tryptophan = 2.99 %      |
| **Y** | Tyrosine = 2.14 %       |



In [1]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Example sequence
sequence = "MIPEFSKSLARTSGLAQVASQCWRMGTEKGHQLPGLSSLWGPAVKVLESGPLLLLILSLGLARAQETLEDVPVQPGFDAQKVEGRWLTVQLAASHAPLAAPDDPLRLALHSIWSRGEDVELVLFWTGEGVCQGLNVTVHPTGLLGQYQSAFKGGGTILLHFVSTDYSHLILYVRFQDDGEVTSLWALLARRMLEDPQWLGQYLAYVSKFHLQEGPVFNLDDQCPPPEASVGATP"

# Create a ProteinAnalysis object
analysis = ProteinAnalysis(sequence)

# Get amino acid counts
aa_counts = analysis.count_amino_acids()

# Convert counts to percentages
total = sum(aa_counts.values())
aa_percentages = {aa: (count / total) * 100 for aa, count in aa_counts.items()}

# Print
for aa, pct in aa_percentages.items():
    print(f"{aa}: {pct:.2f}%")

A: 8.12%
C: 1.28%
D: 4.70%
E: 5.56%
F: 3.42%
G: 9.40%
H: 2.99%
I: 2.14%
K: 2.56%
L: 15.81%
M: 1.28%
N: 0.85%
P: 6.84%
Q: 6.41%
R: 3.85%
S: 7.26%
T: 4.70%
V: 7.69%
W: 2.99%
Y: 2.14%


#### Run FVA

In [2]:
base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" / "Results" 
file_path = base_dir / "FVA_analysis" 

In [14]:
from cobra.flux_analysis import flux_variability_analysis
# Perform FVA for all reactions
fva_results = flux_variability_analysis(model, fraction_of_optimum=0.9)

# Display the results as a DataFrame
fva_df = fva_results.reset_index()
fva_df.columns = ["Reaction", "Minimum Flux", "Maximum Flux"]

base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" / "Data" 
file_path = base_dir / "FVA_analysis" 
#fva_df.to_excel(file_path/'250707_fva_autotrophic.xlsx')
print(fva_df)

              Reaction  Minimum Flux  Maximum Flux
0        EX_25dkglcn_e          0.00          0.00
1          EX_2ameph_e          0.00          0.00
2      EX_2m35mdntha_e          0.00          0.00
3          EX_2pglyc_e          0.00          0.00
4          EX_34dhbz_e          0.00          0.00
...                ...           ...           ...
2384  FACOAE1836Z9Z12Z          0.00          0.00
2385               NIT          0.00          0.00
2386            FERHYD      -1000.00        984.12
2387            CYTHYD          0.00          0.37
2388            Growth          0.04          0.04

[2389 rows x 3 columns]


Run FVA for a specific reaction

In [10]:
fva_results = flux_variability_analysis(model,reaction_list=('LINO_6_DESA'), fraction_of_optimum=0.9)
fva_df = fva_results.reset_index()
fva_df.columns = ["Reaction", "Minimum Flux", "Maximum Flux"]
print(fva_df)

      Reaction  Minimum Flux  Maximum Flux
0  LINO_6_DESA      0.000015      0.000017


Analyze FVA results

In [15]:
base_dir = Path.home() / "Documents" / "PhD" / "SoF1-GEM" 
file_path = base_dir / "Results" / "FVA_analysis" 
# Load the FVA result Excel file
#fva_df = pd.read_excel(file_path/"250807_fva_BLG_autotrophic.xlsx")

# Optional: Display floats with 2 decimals
pd.options.display.float_format = "{:.2f}".format

# Add a flux span column
fva_df["span"] = fva_df["Maximum Flux"] - fva_df["Minimum Flux"]

# 1. Blocked reactions: both min and max = 0
blocked = fva_df[(fva_df["Minimum Flux"] == 0) & (fva_df["Maximum Flux"] == 0)]
num_blocked = len(blocked)

# 2. Essential reactions: min == max != 0
essential = fva_df[(fva_df["Minimum Flux"] == fva_df["Maximum Flux"]) & (fva_df["Minimum Flux"] != 0)]
num_essential = len(essential)

# 3. Flexible reactions: span > 0
flexible = fva_df[fva_df["span"] > 0]
num_flexible = len(flexible)

# 4. Highly variable reactions (span ≥ 100)
high_variability = fva_df[fva_df["span"] >= 100]
num_high_var = len(high_variability)

# 5. Top 10 most variable reactions
top_10_variable = fva_df.sort_values(by="span", ascending=False).head(10)

# Print summary
print("🔍 Summary of FVA Analysis")
print(f"Total reactions: {len(fva_df)}")
print(f"Blocked reactions: {num_blocked}")
print(f"Essential reactions (fixed flux): {num_essential}")
print(f"Flexible reactions (variable flux): {num_flexible}")
print(f"Highly variable (span ≥ 100): {num_high_var}")
print("\nTop 10 most variable reactions:")
print(top_10_variable[["Reaction", "Minimum Flux", "Maximum Flux", "span"]])

🔍 Summary of FVA Analysis
Total reactions: 2389
Blocked reactions: 891
Essential reactions (fixed flux): 0
Flexible reactions (variable flux): 1496
Highly variable (span ≥ 100): 300

Top 10 most variable reactions:
      Reaction  Minimum Flux  Maximum Flux    span
1327    HBCHLR      -1000.00       1000.00 2000.00
954   ECOAH5_1      -1000.00       1000.00 2000.00
943     ECOAH3      -1000.00       1000.00 2000.00
932   ECOAH1_1      -1000.00       1000.00 2000.00
2035   RECOAH5      -1000.00       1000.00 2000.00
2025   RECOAH2      -1000.00       1000.00 2000.00
609     APAT_1      -1000.00       1000.00 2000.00
1309     HACD4      -1000.00       1000.00 2000.00
803    CYTK2_1      -1000.00       1000.00 2000.00
1681     NDPK1      -1000.00       1000.00 2000.00


In [16]:
fva_df['span'] = fva_df['span'].apply(lambda x: round(x, 6))

In [17]:
fva_df_filtered = fva_df[
    (fva_df['Maximum Flux'] != 0) & #remove unnecessary reactions
    (fva_df['Minimum Flux'] != 0) &
    (fva_df['span'] > 0.1) & # reactions with very low flexibility
    (fva_df['span'] < 1000)
]
fva_df_filtered = fva_df_filtered.sort_values("span",ascending=False,ignore_index=True) # reorder by the span from the highest to the lowest
fva_df_filtered.to_excel(file_path/'250808_fva_filtered_autotrophic.xlsx')

In [11]:
fva_df_filtered

,Reaction,Minimum Flux,Maximum Flux,span
0,r2465_1,-2.59,-0.00,2.59
1,NH3c,-0.68,0.66,1.34
2,ORNTAC,-0.65,0.67,1.31
3,ACOTA,-0.67,0.65,1.31
4,ORNTA,-0.67,0.65,1.31
...,...,...,...,...
141,CLPNS161pp,0.00,0.11,0.11
142,PYDXO,-0.11,-0.00,0.11
143,ACPPAT181,0.00,0.11,0.11
144,AGPAT181,0.00,0.11,0.11


remove loop reactions

In [ ]:
loop_reactions = [model.reactions.FRD7, model.reactions.SUCDi]
flux_variability_analysis(model, reaction_list=loop_reactions, loopless=False)